# AI381 Lab 1 · Baseline churn model
This is a typical course notebook: it works on one laptop, once. Your job in Lab 1 is to deliver it by hand and log what that costs you.

**Start your stopwatch now.**

In [16]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import f1_score, roc_auc_score
import joblib

In [17]:
DATA = 'churn_v1.csv'   # Twist 1: change this to churn_v2.csv
df = pd.read_csv(DATA)
df.head()

,customer_id,tenure_months,monthly_charges,contract,internet_service,support_calls,churn
0,C100000,35,121.08,month-to-month,fiber,1,1
1,C100001,71,70.56,month-to-month,dsl,0,0
2,C100002,23,102.54,one-year,dsl,1,0
3,C100003,32,76.55,month-to-month,dsl,3,1
4,C100004,37,92.74,month-to-month,fiber,0,0


In [18]:
numeric = ['tenure_months', 'monthly_charges', 'support_calls']
categorical = ['contract', 'internet_service']
X = df[numeric + categorical]
y = df['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [19]:
model = Pipeline([
    ('prep', ColumnTransformer([
        ('num', StandardScaler(), numeric),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
    ])),
    ('clf', LogisticRegression(max_iter=1000)),
])
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
print('F1 :', round(f1_score(y_test, proba > 0.5), 3))
print('AUC:', round(roc_auc_score(y_test, proba), 3))

F1 : 0.512
AUC: 0.783


In [20]:
joblib.dump(model, 'model.joblib')
print('saved model.joblib')

saved model.joblib


Next: start the API with `uvicorn app.main:app --port 8000` and send three predictions with curl (see README).